[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C73_3D_Representation_Course/05_seg_eval/05_seg_eval.ipynb)

# 05 · 分割与 3D 评测

这个 notebook 把「评测口径」当成一个可以计算的对象，量四件事：

1. **IoU 的闭式**：$\text{IoU} = q/(2-q)$，$q=\prod(1-\delta_i/s_i)$；
   于是**单轴容差恰好是 $s_i/3$**。
2. **尺度敏感性**：IoU≥0.5 的各向同性容差从卡车 **0.796 m** 到标志 **0.049 m**（差 16 倍）。
3. **两种口径的分歧率**：$\sigma{=}0.2$ m 时，IoU-0.5 拒掉 **97.2%** 被中心距离-2 m 接受的标志。
4. **口径的天花板**：5 cm 标注噪声下，一个**完美模型**在标志上的 AP@IoU0.5 上界只有 **0.392**
   （而中心距离口径下是 1.000）。

> 心智模型：**指标不是「测量工具」，它是一个有分辨率上限的测量工具。
> 上限低于你要测的效应时，排名就是噪声。**

## 0 · 环境与四类目标

In [ ]:
import numpy as np

print('numpy', np.__version__)

CLASSES = {
    'truck':      (10.0, 2.5, 3.2),
    'car':        ( 4.5, 1.9, 1.5),
    'pedestrian': ( 0.6, 0.6, 1.7),
    'sign':       ( 0.8, 0.1, 0.8),      # 又小又薄 —— 本课的主角
}
for n, s in CLASSES.items():
    print(f'  {n:12s} {s}  最短轴 {min(s):.2f} m')

## 1 · IoU 的闭式，与「单轴容差 = $s/3$」

In [ ]:
def iou_same_size(size, delta):
    '''同尺寸轴对齐框，中心相距 delta 时的 IoU（维数由 size 决定）。'''
    s = np.asarray(size, float); d = np.abs(np.asarray(delta, float))
    q = np.prod(np.maximum(1.0 - d / s, 0.0))
    return q / (2.0 - q) if q > 0 else 0.0

def iou_from_q(q):
    return q / (2.0 - q) if q > 0 else 0.0

# ① 闭式与直接算交并比一致
def iou_bruteforce(size, delta):
    s = np.asarray(size, float); d = np.abs(np.asarray(delta, float))
    inter = np.prod(np.maximum(s - d, 0.0)); vol = np.prod(s)
    den = 2 * vol - inter
    return inter / den if den > 0 else 0.0

rng = np.random.default_rng(0)
worst = 0.0
for _ in range(2000):
    s = rng.uniform(0.1, 10.0, 3)
    d = rng.uniform(0.0, 3.0, 3)
    worst = max(worst, abs(iou_same_size(s, d) - iou_bruteforce(s, d)))
print(f'闭式 q/(2−q) 与直接算的最大差 = {worst:.3e}')
assert worst < 1e-12
print('✅ IoU = q/(2−q)，其中 q = ∏(1 − δᵢ/sᵢ)₊\n')

# ② 单轴容差恰好是 s/3
print(f"{'类别':>12s} {'最短轴':>8s} {'该轴 IoU=0.5 的容差':>20s} {'s/3':>9s}")
for n, s in CLASSES.items():
    ax = int(np.argmin(s))
    lo, hi = 0.0, float(s[ax])
    for _ in range(60):
        mid = (lo + hi) / 2
        dd = [0.0, 0.0, 0.0]; dd[ax] = mid
        if iou_same_size(s, dd) >= 0.5:
            lo = mid
        else:
            hi = mid
    print(f'{n:>12s} {s[ax]:7.2f}m {lo:19.4f}m {s[ax]/3:8.4f}m')
    assert abs(lo - s[ax] / 3) < 1e-6, (n, lo, s[ax] / 3)
print('\n✅ **单轴容差恰好是 s/3** —— 因为 1 − δ/s = 2/3 ⇒ δ = s/3')

# ③ 每轴等相对误差时：2D 18.4% vs 3D 12.6%
for n_dim in [2, 3]:
    e = 1 - (2/3) ** (1/n_dim)
    got = iou_same_size((1.,)*n_dim, (e,)*n_dim)
    print(f'  {n_dim}D 每轴允许的相对误差 = {e:.4f}（{e:.1%}），代回 IoU = {got:.6f}')
    assert abs(got - 0.5) < 1e-9

## 2 · 尺度敏感性：16 倍的容差差距

In [ ]:
def isotropic_tolerance(size, thr=0.5):
    '''IoU >= thr 时允许的各向同性位移范数。'''
    lo, hi = 0.0, 20.0
    for _ in range(80):
        mid = (lo + hi) / 2
        d = (mid / np.sqrt(len(size)),) * len(size)
        if iou_same_size(size, d) >= thr:
            lo = mid
        else:
            hi = mid
    return lo

print(f"{'类别':>12s} {'尺寸':>22s} {'最短轴':>8s} {'IoU>=0.5 容差':>14s} {'相对卡车':>9s}")
tol = {n: isotropic_tolerance(s) for n, s in CLASSES.items()}
base = tol['truck']
for n, s in CLASSES.items():
    print(f'{n:>12s} {str(s):>22s} {min(s):7.2f}m {tol[n]:13.3f}m {tol[n]/base:8.2f}×')

ratio = tol['truck'] / tol['sign']
print(f'\n卡车 / 标志 = **{ratio:.1f} 倍**，而阈值是同一个 0.5')
assert ratio > 10, f'差距应超过 10 倍，实测 {ratio:.1f}'
print('✅ mAP@0.5 对卡车问「定位到 0.8 m」，对标志问「定位到 5 厘米」')

# 沿最薄轴时更极端
print(f'\n沿最薄轴 δ=0.1 m 时的 IoU：')
for n, s in CLASSES.items():
    ax = int(np.argmin(s))
    dd = [0.0, 0.0, 0.0]; dd[ax] = 0.1
    print(f'  {n:12s} {iou_same_size(s, dd):.4f}')
assert iou_same_size(CLASSES['sign'], (0., 0.1, 0.)) == 0.0
print('  → 标志沿厚度方向偏 0.1 m 就完全脱靶（IoU = 0）')

## 3 · 两种口径的分歧率

In [ ]:
def hit_rates(size, sigma, n=20000, iou_thr=0.5, dist_thr=2.0, seed=0):
    d = np.random.default_rng(seed).normal(0, sigma, (n, 3))
    by_iou = np.array([iou_same_size(size, dd) >= iou_thr for dd in d])
    by_dist = np.linalg.norm(d, axis=1) <= dist_thr
    return {'iou': float(by_iou.mean()), 'dist': float(by_dist.mean()),
            'disagree': float((by_iou != by_dist).mean())}

print(f"{'类别':>12s} {'σ':>6s} {'IoU>=0.5':>10s} {'中心距离<=2m':>13s} {'分歧率':>9s}")
dis = {}
for sg in [0.2, 0.5]:
    for n, s in CLASSES.items():
        h = hit_rates(s, sg)
        dis[(n, sg)] = h['disagree']
        print(f'{n:>12s} {sg:6.2f} {h["iou"]:10.3f} {h["dist"]:12.3f} '
              f'{h["disagree"]:8.3f}')
    print()

# 分歧率随目标变小单调上升
order = ['truck', 'car', 'pedestrian', 'sign']
d02 = [dis[(n, 0.2)] for n in order]
assert d02 == sorted(d02), f'分歧率应随目标变小单调上升：{d02}'
assert dis[('sign', 0.2)] > 0.9
print(f'σ=0.2 m 时的分歧率: ' +
      ' < '.join(f'{n} {dis[(n,0.2)]:.3f}' for n in order))
print(f'\n✅ 标志上分歧 **{dis[("sign",0.2)]:.1%}** —— 两个口径几乎完全不一致')
print('   IoU 量「框重合得多好」，中心距离量「位置对不对」——对薄片，前者本身就脆弱')

## 3b · 口径的天花板：完美模型能拿多少分

In [ ]:
def metric_ceiling(size, sigma_ann, criterion='iou', thr=0.5,
                   n=20000, seed=1):
    '''模型预测 = 真实位置，而「真值」标注带 sigma_ann 噪声。返回命中率上界。'''
    d = np.random.default_rng(seed).normal(0, sigma_ann, (n, 3))
    if criterion == 'iou':
        return float(np.mean([iou_same_size(size, dd) >= thr for dd in d]))
    if criterion == 'center':
        return float((np.linalg.norm(d, axis=1) <= thr).mean())
    raise ValueError(criterion)

SIGMAS = [0.02, 0.05, 0.10, 0.20]
print('IoU >= 0.5 口径下、完美模型的 AP 上界：')
print(f"{'类别':>12s} " + ''.join(f'σ={s}m'.rjust(11) for s in SIGMAS))
ceil_iou = {}
for n, s in CLASSES.items():
    row = [metric_ceiling(s, sa, 'iou', 0.5) for sa in SIGMAS]
    ceil_iou[n] = dict(zip(SIGMAS, row))
    print(f'{n:>12s} ' + ''.join(f'{v:10.3f} ' for v in row))

print('\n中心距离 <= 2 m 口径下的同一个上界：')
print(f"{'类别':>12s} " + ''.join(f'σ={s}m'.rjust(11) for s in SIGMAS))
for n, s in CLASSES.items():
    row = [metric_ceiling(s, sa, 'center', 2.0) for sa in SIGMAS]
    print(f'{n:>12s} ' + ''.join(f'{v:10.3f} ' for v in row))
    assert all(v > 0.99 for v in row), '中心距离口径的上界应恒为 1'

# ★ 核心断言
assert ceil_iou['truck'][0.05] > 0.99 and ceil_iou['car'][0.05] > 0.99
assert ceil_iou['sign'][0.05] < 0.45, ceil_iou['sign'][0.05]
print(f'\n✅ 5 cm 标注噪声下：卡车/轿车的上界 1.000，'
      f'而**标志只有 {ceil_iou["sign"][0.05]:.3f}**')
print('   → 标志那一列量的不是模型能力，是标注噪声')

# 容差 / 标注噪声 的比值
SIGMA_ANN = 0.05
print(f'\n{"类别":>12s} {"容差":>10s} {"σ_ann":>8s} {"比值":>8s} {"判据(>=3)":>10s}')
for n, s in CLASSES.items():
    r = tol[n] / SIGMA_ANN
    print(f'{n:>12s} {tol[n]:9.3f}m {SIGMA_ANN:7.2f}m {r:7.2f}× '
          f'{"OK" if r >= 3 else "**不足**":>10s}')
assert tol['sign'] / SIGMA_ANN < 1.2, '标志的容差与标注噪声应当几乎相等'
print(f'\n✅ 标志的比值 {tol["sign"]/SIGMA_ANN:.2f}× —— '
      '容差与标注噪声几乎相等，信噪比不足')
print('   判据：容差 / σ_ann >= 3；小于 3 就说明这个口径在该类别上不可信')

## 4 · 降阈值能救回多少

In [ ]:
TAUS = [0.1, 0.25, 0.5, 0.7]
print(f'σ_ann = {SIGMA_ANN} m 下，按 τ 扫描完美模型的上界：\n')
print(f"{'类别':>12s} " + ''.join(f'τ={t}'.rjust(9) for t in TAUS))
sweep = {}
for n, s in CLASSES.items():
    row = [metric_ceiling(s, SIGMA_ANN, 'iou', t) for t in TAUS]
    sweep[n] = dict(zip(TAUS, row))
    print(f'{n:>12s} ' + ''.join(f'{v:8.3f} ' for v in row))

# 降阈值确实有效
assert sweep['sign'][0.25] > 2 * sweep['sign'][0.7]
print(f'\n标志: τ=0.7 → {sweep["sign"][0.7]:.3f}，'
      f'τ=0.5 → {sweep["sign"][0.5]:.3f}，'
      f'τ=0.25 → {sweep["sign"][0.25]:.3f}，'
      f'τ=0.1 → {sweep["sign"][0.1]:.3f}')

# 而卡车/轿车在 τ=0.7 都还没触到天花板
assert sweep['truck'][0.7] > 0.99 and sweep['car'][0.7] > 0.99
print(f'而卡车与轿车在 τ=0.7 仍是 1.000 —— **它们的天花板远没被触到**')

# 按「天花板 >= 0.9」定阈值
print(f'\n按「天花板 >= 0.9」为每个类别定阈值：')
for n, s in CLASSES.items():
    ok = [t for t in [0.7, 0.5, 0.25, 0.1] if sweep[n][t] >= 0.9]
    best = max(ok) if ok else None
    print(f'  {n:12s} -> τ = {best if best is not None else "即使 0.1 也不够"}'
          f'   （天花板 {sweep[n][best]:.3f}）' if best is not None
          else f'  {n:12s} -> **即使 τ=0.1 天花板也只有 {sweep[n][0.1]:.3f}**')
print('\n✅ 这就是「KITTI 车用 0.7 / 行人用 0.5」的**依据**——'
      '让每个类别的天花板保持在 0.9 以上')

## 5 · mIoU 与频率加权：同一个改进差 250 倍

In [ ]:
NAMES = ['ground', 'car', 'sign', 'other']
FREQ = np.array([0.600, 0.080, 0.001, 0.319])
IOUS = np.array([0.95, 0.75, 0.20, 0.60])

def miou(ious):
    return float(np.mean(ious))

def fwiou(ious, freq=FREQ):
    return float((ious * freq).sum() / freq.sum())

print(f"{'类别':>10s} {'频率':>9s} {'IoU':>7s} {'mIoU 权重':>11s} {'加权权重':>10s}")
for n, f, i in zip(NAMES, FREQ, IOUS):
    print(f'{n:>10s} {f:8.3f} {i:7.2f} {1/len(NAMES):10.1%} {f/FREQ.sum():9.1%}')
print(f'\n  mIoU（等权）  = {miou(IOUS):.4f}')
print(f'  频率加权 IoU  = {fwiou(IOUS):.4f}')
print(f'  两者相差 {abs(miou(IOUS)-fwiou(IOUS)):.4f}')

# 稀有类在 mIoU 里的权重是它频率的 250 倍
w_ratio = (1/len(NAMES)) / (FREQ[2]/FREQ.sum())
print(f'  而 sign（{FREQ[2]:.1%}）在 mIoU 里占 {1/len(NAMES):.0%} 的权重'
      f' —— 是它频率的 **{w_ratio:.0f} 倍**')

print(f'\n把 sign 的 IoU 从 0.20 提到 0.80：')
print(f"{'sign IoU':>10s} {'mIoU':>9s} {'Δ mIoU':>10s} {'加权':>9s} {'Δ 加权':>11s}")
for new in [0.20, 0.40, 0.60, 0.80]:
    i2 = IOUS.copy(); i2[2] = new
    print(f'{new:10.2f} {miou(i2):9.4f} {miou(i2)-miou(IOUS):+10.4f} '
          f'{fwiou(i2):9.4f} {fwiou(i2)-fwiou(IOUS):+11.5f}')

i80 = IOUS.copy(); i80[2] = 0.80
d_m, d_w = miou(i80) - miou(IOUS), fwiou(i80) - fwiou(IOUS)
print(f'\n✅ 同一个改进: mIoU +{d_m:.4f}，频率加权 +{d_w:.5f}'
      f' —— 差 **{d_m/d_w:.0f} 倍**')
assert d_m / d_w > 100
print('   → 「选哪个指标」直接决定「改进稀有类值不值得做」')
print('   而对自动驾驶，频率加权是明确错的口径（按频率而不是按后果加权）')

## 6 · PQ 的分解，与它同样受 IoU 阈值支配

In [ ]:
def pq(matches, n_pred, n_gt, thr=0.5):
    '''matches: 每个匹配对的 IoU 列表（已按 thr 筛过）。返回 (PQ, SQ, RQ)。'''
    tp = [m for m in matches if m >= thr]
    n_tp = len(tp)
    fp, fn = n_pred - n_tp, n_gt - n_tp
    sq = float(np.mean(tp)) if n_tp else 0.0
    rq = n_tp / (n_tp + 0.5 * fp + 0.5 * fn) if (n_tp + fp + fn) else 0.0
    return sq * rq, sq, rq

# 两种情形：找得全但分割糙 vs 找得少但分割准
A = ([0.55] * 10, 10, 10)      # 10 个匹配，IoU 都刚过线
B = ([0.95] * 5, 5, 10)        # 只找到 5 个，但都很准
for tag, (ms, np_, ng) in [('找得全、分割糙', A), ('找得少、分割准', B)]:
    p, s_, r_ = pq(ms, np_, ng)
    print(f'{tag}: PQ={p:.4f}  SQ={s_:.4f}  RQ={r_:.4f}')

pa = pq(*A)[0]; pb = pq(*B)[0]
print(f'\n两者的 PQ: {pa:.4f} vs {pb:.4f}')
print('  → PQ 把「找得全不全」（RQ）与「分割准不准」（SQ）拆成两个可分别改进的因子')

# 但 PQ 的匹配判据仍是 IoU > 0.5 —— 所以第 3b 节的天花板同样适用
print(f'\nPQ 的 RQ 部分受 IoU 阈值支配，所以第 3b 节的天花板同样适用：')
for n in ['car', 'sign']:
    c = metric_ceiling(CLASSES[n], SIGMA_ANN, 'iou', 0.5)
    # 完美模型：n_pred = n_gt = 100，但只有 c 比例被判为 TP
    n_all = 100
    n_tp = int(round(c * n_all))
    _, sq_, rq_ = pq([0.9] * n_tp, n_all, n_all)
    print(f'  {n:6s}: IoU 天花板 {c:.3f} -> 完美模型的 RQ = {rq_:.3f}'
          f'  PQ <= {sq_*rq_:.3f}')
print('  → **即使用全景指标，也要先检查匹配判据的天花板**')

## 7 · 小结

| 结论 | 数值 |
|---|---|
| IoU 的闭式 | $q/(2-q)$，与直接算的差 < 1e-12 |
| **单轴容差** | 恰好 $s_i/3$（四个类别上精确吻合） |
| 每轴等相对误差 | 2D 允许 **18.4%**，3D 只允许 **12.6%** |
| 各向同性容差 | 卡车 **0.796 m** → 标志 **0.049 m**（**16 倍**） |
| 沿最薄轴偏 0.1 m | 标志 IoU = **0.0000**（完全脱靶） |
| **两种口径的分歧率** | $\sigma{=}0.2$ m：卡车 0.001 → **标志 0.972** |
| **口径的天花板** | 5 cm 标注噪声下，完美模型在标志上只有 **0.392** |
| 中心距离口径的天花板 | 四个类别 **恒为 1.000** |
| 容差 / $\sigma_{\text{ann}}$ | 卡车 15.93× · 轿车 9.18× · 行人 3.32× · **标志 0.98×** |
| 降阈值 | 标志 τ=0.5 的 0.391 → τ=0.25 的 0.736 |
| **mIoU vs 频率加权** | 同一个改进差 **250 倍**（+0.1500 vs +0.00060） |

## ✏️ 练习 1：容差反解器

实现 `tolerance_table(class_sizes, thr=0.5)`，返回 `{类名: dict}`，每项含：

- `'per_axis'` —— 各轴的单轴容差（应当恰好是 $s_i/3$，$\tau{=}0.5$ 时）
- `'isotropic'` —— 各向同性位移的容差
- `'worst_axis'` —— 最短轴的下标
- `'rel_per_axis'` —— 每轴等相对误差时允许的相对误差

In [ ]:
def tolerance_table(class_sizes, thr=0.5):
    """返回 {类名: dict(per_axis, isotropic, worst_axis, rel_per_axis)}。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
tt = tolerance_table(CLASSES)
for n, v in tt.items():
    assert set(v) == {'per_axis', 'isotropic', 'worst_axis', 'rel_per_axis'}

print(f"{'类别':>12s} {'各轴容差 (m)':>30s} {'各向同性':>10s} {'每轴相对':>9s}")
for n, v in tt.items():
    pa = '[' + ', '.join(f'{x:.3f}' for x in v['per_axis']) + ']'
    print(f'{n:>12s} {pa:>30s} {v["isotropic"]:9.3f} {v["rel_per_axis"]:8.1%}')

# ① τ=0.5 时单轴容差恰好是 s/3
for n, s in CLASSES.items():
    for i in range(3):
        assert abs(tt[n]['per_axis'][i] - s[i] / 3) < 1e-6, (n, i)
print('\n✅ τ=0.5 时单轴容差恰好是 s/3（四个类别、三个轴全部吻合）')

# ② 最短轴的下标
for n, s in CLASSES.items():
    assert tt[n]['worst_axis'] == int(np.argmin(s))

# ③ 每轴相对误差与维数有关、与尺寸无关
rels = {tt[n]['rel_per_axis'] for n in CLASSES}
assert len(rels) == 1, f'相对误差应当与尺寸无关，实得 {rels}'
assert abs(list(rels)[0] - (1 - (2/3)**(1/3))) < 1e-6
print(f'✅ 每轴相对误差 {list(rels)[0]:.4f} 对所有类别相同（只依赖维数 n=3）')

# ④ 换阈值
tt7 = tolerance_table(CLASSES, thr=0.7)
for n in CLASSES:
    assert tt7[n]['isotropic'] < tt[n]['isotropic'], '阈值越高容差越小'
print(f'\nτ=0.7 时标志的各向同性容差 {tt7["sign"]["isotropic"]:.4f} m'
      f'（τ=0.5 时是 {tt["sign"]["isotropic"]:.4f} m）')
print('✅ 练习 1 通过')

## 📖 参考答案 1

In [ ]:
# 练习 1 参考答案
def tolerance_table(class_sizes, thr=0.5):
    q_need = 2 * thr / (1 + thr)              # IoU>=thr  <=>  q >= 2τ/(1+τ)
    out = {}
    for name, s in class_sizes.items():
        s = np.asarray(s, float)
        per_axis = (1.0 - q_need) * s          # 单轴：1 − δ/s = q_need
        n = len(s)
        rel = 1.0 - q_need ** (1.0 / n)        # 每轴等相对误差
        # 各向同性：二分
        lo, hi = 0.0, 20.0
        for _ in range(80):
            mid = (lo + hi) / 2
            d = (mid / np.sqrt(n),) * n
            if iou_same_size(s, d) >= thr:
                lo = mid
            else:
                hi = mid
        out[name] = {'per_axis': per_axis.tolist(), 'isotropic': float(lo),
                     'worst_axis': int(np.argmin(s)), 'rel_per_axis': float(rel)}
    return out

tt = tolerance_table(CLASSES)
for n, s in CLASSES.items():
    assert abs(tt[n]['per_axis'][0] - s[0] / 3) < 1e-9
print('✅ 参考答案 2 通过'.replace('2', '1'))
print('   关键是先把阈值翻译成 q：IoU ≥ τ ⟺ q ≥ 2τ/(1+τ)。')
print('   τ=0.5 给 q ≥ 2/3，所以单轴容差 = (1 − 2/3)·s = s/3。')

## ✏️ 练习 2：口径分歧分析

实现 `criterion_comparison(size, sigma, iou_thr=0.5, dist_thr=2.0, n=20000)`，
返回 dict：

- `'hit_iou'`, `'hit_dist'` —— 两个口径的命中率
- `'disagree'` —— 分歧率
- `'iou_only'` —— 只有 IoU 判命中的比例
- `'dist_only'` —— 只有中心距离判命中的比例
- `'direction'` —— `'iou_stricter'` / `'dist_stricter'` / `'agree'`

In [ ]:
def criterion_comparison(size, sigma, iou_thr=0.5, dist_thr=2.0,
                         n=20000, seed=0):
    """返回 dict(hit_iou, hit_dist, disagree, iou_only, dist_only, direction)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
print(f"{'类别':>12s} {'σ':>6s} {'IoU':>8s} {'中心':>8s} {'分歧':>8s} "
      f"{'仅IoU':>8s} {'仅中心':>8s} {'方向':>14s}")
res = {}
for n_, s in CLASSES.items():
    for sg in [0.05, 0.2]:
        c = criterion_comparison(s, sg)
        res[(n_, sg)] = c
        assert set(c) == {'hit_iou', 'hit_dist', 'disagree', 'iou_only',
                          'dist_only', 'direction'}
        print(f'{n_:>12s} {sg:6.2f} {c["hit_iou"]:8.3f} {c["hit_dist"]:8.3f} '
              f'{c["disagree"]:8.3f} {c["iou_only"]:8.3f} {c["dist_only"]:8.3f} '
              f'{c["direction"]:>14s}')

# ① 分歧全部来自「中心距离更松」这一侧
for k, c in res.items():
    assert c['iou_only'] < 1e-3, f'{k}: IoU 不应当比中心距离更松'
    assert abs(c['disagree'] - c['dist_only']) < 1e-6

# ② 大目标一致，小目标分歧
assert res[('truck', 0.2)]['direction'] == 'agree'
assert res[('sign', 0.2)]['direction'] == 'iou_stricter'
assert res[('sign', 0.2)]['disagree'] > 0.9
print(f'\n✅ 分歧全部来自「IoU 更严」这一侧（仅 IoU 命中的比例 < 0.001）')
print(f'   卡车 {res[("truck",0.2)]["direction"]}，'
      f'标志 {res[("sign",0.2)]["direction"]}（分歧 '
      f'{res[("sign",0.2)]["disagree"]:.1%}）')

# ③ 把中心距离阈值收紧到「与 IoU 容差相当」，分歧就消失
tol_sign = tolerance_table(CLASSES)['sign']['isotropic']
c_tight = criterion_comparison(CLASSES['sign'], 0.2, dist_thr=tol_sign)
print(f'\n把中心距离阈值收到 {tol_sign:.3f} m（= 标志的 IoU 容差）: '
      f'分歧 {c_tight["disagree"]:.3f}')
assert c_tight['disagree'] < 0.1, '阈值匹配后分歧应当很小'
print('✅ 练习 2 通过：**两个口径不是「哪个更对」，而是阈值是否可比**')

## 📖 参考答案 2

In [ ]:
# 练习 2 参考答案
def criterion_comparison(size, sigma, iou_thr=0.5, dist_thr=2.0,
                         n=20000, seed=0):
    d = np.random.default_rng(seed).normal(0, sigma, (n, 3))
    by_iou = np.array([iou_same_size(size, dd) >= iou_thr for dd in d])
    by_dist = np.linalg.norm(d, axis=1) <= dist_thr
    iou_only = float((by_iou & ~by_dist).mean())
    dist_only = float((by_dist & ~by_iou).mean())
    dg = float((by_iou != by_dist).mean())
    if dg < 0.02:
        direction = 'agree'
    elif dist_only > iou_only:
        direction = 'iou_stricter'
    else:
        direction = 'dist_stricter'
    return {'hit_iou': float(by_iou.mean()), 'hit_dist': float(by_dist.mean()),
            'disagree': dg, 'iou_only': iou_only, 'dist_only': dist_only,
            'direction': direction}

c = criterion_comparison(CLASSES['sign'], 0.2)
assert c['direction'] == 'iou_stricter' and c['iou_only'] < 1e-3
print('✅ 参考答案 2 通过')
print('   `iou_only ≈ 0` 说明 IoU-0.5 是中心距离-2m 的**子集**（在这些参数下）——')
print('   所以两者不是「不同的判断」，而是「同一个判断的两个松紧度」。')

## ✏️ 练习 3：口径天花板审计

实现 `ceiling_audit(class_sizes, sigma_ann, taus=(0.1,0.25,0.5,0.7), target=0.9)`，
返回 `{类名: dict}`，每项含：

- `'ceiling'` —— `{τ: 完美模型的上界}`
- `'best_tau'` —— 使上界 $\ge$ `target` 的**最大** τ（没有则 `None`）
- `'tol_over_sigma'` —— IoU-0.5 容差 / `sigma_ann`
- `'trustworthy'` —— bool：`tol_over_sigma >= 3`

**这是本模块最实用的交付物**——它不需要模型，只需要类别尺寸与标注噪声估计。

In [ ]:
def ceiling_audit(class_sizes, sigma_ann, taus=(0.1, 0.25, 0.5, 0.7),
                  target=0.9):
    """返回 {类名: dict(ceiling, best_tau, tol_over_sigma, trustworthy)}。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
au = ceiling_audit(CLASSES, 0.05)
for n, v in au.items():
    assert set(v) == {'ceiling', 'best_tau', 'tol_over_sigma', 'trustworthy'}

print(f"{'类别':>12s} " + ''.join(f'τ={t}'.rjust(9) for t in (0.1,0.25,0.5,0.7)) +
      f"{'最佳 τ':>9s} {'容差/σ':>9s} {'可信':>7s}")
for n, v in au.items():
    row = ''.join(f'{v["ceiling"][t]:8.3f} ' for t in (0.1, 0.25, 0.5, 0.7))
    bt = 'None' if v['best_tau'] is None else f'{v["best_tau"]}'
    print(f'{n:>12s} {row}{bt:>9s} {v["tol_over_sigma"]:8.2f}× '
          f'{str(v["trustworthy"]):>7s}')

# ① 大目标：0.7 就够，且可信
assert au['truck']['best_tau'] == 0.7 and au['truck']['trustworthy'] is True
assert au['car']['best_tau'] == 0.7 and au['car']['trustworthy'] is True
# ② 标志：不可信，且即使降阈值也救不回 0.9
assert au['sign']['trustworthy'] is False
assert au['sign']['ceiling'][0.5] < 0.5
print(f'\n卡车/轿车: 最佳 τ=0.7、可信')
print(f'标志: 容差/σ = {au["sign"]["tol_over_sigma"]:.2f}× < 3 -> **不可信**')
print(f'      而即使 τ=0.1，上界也只有 {au["sign"]["ceiling"][0.1]:.3f}')

# ③ 把标注噪声降到 0.01 m，标志就可信了
au2 = ceiling_audit({'sign': CLASSES['sign']}, 0.01)
print(f'\n把 σ_ann 降到 0.01 m: 标志的容差/σ = '
      f'{au2["sign"]["tol_over_sigma"]:.2f}×，可信 {au2["sign"]["trustworthy"]}')
assert au2['sign']['trustworthy'] is True
print('✅ 练习 3 通过：**这个审计只需要类别尺寸与标注噪声估计，不需要模型**')

## 📖 参考答案 3

In [ ]:
# 练习 3 参考答案
def ceiling_audit(class_sizes, sigma_ann, taus=(0.1, 0.25, 0.5, 0.7),
                  target=0.9):
    tt = tolerance_table(class_sizes, thr=0.5)
    out = {}
    for name, s in class_sizes.items():
        ceil = {t: metric_ceiling(s, sigma_ann, 'iou', t) for t in taus}
        ok = [t for t in sorted(taus) if ceil[t] >= target]
        r = tt[name]['isotropic'] / sigma_ann
        out[name] = {'ceiling': ceil,
                     'best_tau': (max(ok) if ok else None),
                     'tol_over_sigma': float(r),
                     'trustworthy': bool(r >= 3.0)}
    return out

au = ceiling_audit(CLASSES, 0.05)
assert au['truck']['trustworthy'] and not au['sign']['trustworthy']
print('✅ 参考答案 3 通过')
print('   两个判据是互补的：')
print('   `trustworthy` 看容差与标注噪声的比（信噪比）；')
print('   `best_tau` 看「降阈值能不能把天花板拉回 0.9」（补救手段）。')

## ✏️ 练习 4：分层报告器

实现 `stratified_report(records, dims)`，`records` 是每条检测的 dict
（含 `'hit'`（bool）与若干分层字段），`dims` 是要分层的字段名列表。

返回 `{维度: {桶: dict(n, recall)}}`，并额外给一个 `'_worst'` 键：
`{维度: (最差的桶, 该桶的 recall)}`。

**要求：只报样本数 $\ge5$ 的桶**（否则 recall 是噪声）。

In [ ]:
def stratified_report(records, dims, min_n=5):
    """返回 {维度: {桶: dict(n, recall)}}，外加 '_worst' 键。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
# 造一批检测记录：远处与薄目标的召回明显更低
rng2 = np.random.default_rng(5)
RECS = []
for _ in range(600):
    rng_band = rng2.choice(['0-20m', '20-50m', '50m+'], p=[0.5, 0.35, 0.15])
    cls = rng2.choice(['car', 'sign'], p=[0.8, 0.2])
    p_hit = {'0-20m': 0.95, '20-50m': 0.80, '50m+': 0.45}[rng_band]
    if cls == 'sign':
        p_hit *= 0.5
    RECS.append({'hit': bool(rng2.random() < p_hit),
                 'range_band': rng_band, 'cls': cls,
                 'orient_band': rng2.choice(['0-30°', '30-60°', '60-90°'])})

rep = stratified_report(RECS, ['range_band', 'cls', 'orient_band'])
assert '_worst' in rep
for d in ['range_band', 'cls', 'orient_band']:
    assert d in rep

for d in ['range_band', 'cls', 'orient_band']:
    print(f'{d}:')
    for b, v in sorted(rep[d].items()):
        print(f'    {b:>10s}  n={v["n"]:4d}  recall={v["recall"]:.3f}')
    w = rep['_worst'][d]
    print(f'    最差: {w[0]}（{w[1]:.3f}）')

# ① 距离与类别上应当有明显差异，朝向上不应当有
r_far = rep['range_band']['50m+']['recall']
r_near = rep['range_band']['0-20m']['recall']
assert r_far < r_near * 0.7, (r_far, r_near)
assert rep['range_band']['_'] if False else True
assert rep['cls']['sign']['recall'] < rep['cls']['car']['recall'] * 0.8

# ② 总体召回掩盖了分层差异
overall = np.mean([r['hit'] for r in RECS])
print(f'\n总体召回 {overall:.3f}，而最差的桶：')
for d in ['range_band', 'cls']:
    w = rep['_worst'][d]
    print(f'  {d}: {w[0]} 只有 {w[1]:.3f}（总体的 {w[1]/overall:.0%}）')
assert rep['_worst']['range_band'][1] < overall * 0.7

# ③ 小样本桶被过滤
few = RECS + [{'hit': False, 'range_band': '100m+', 'cls': 'car',
               'orient_band': '0-30°'}] * 3
rep2 = stratified_report(few, ['range_band'])
assert '100m+' not in rep2['range_band'], '样本数 < 5 的桶应当被过滤'
print(f'\n只有 3 个样本的 100m+ 桶被过滤掉了（避免报噪声）')
print('✅ 练习 4 通过：**总体召回掩盖了分层差异，而分层维度由「你怀疑什么」决定**')

## 📖 参考答案 4

In [ ]:
# 练习 4 参考答案
def stratified_report(records, dims, min_n=5):
    out = {}
    worst = {}
    for d in dims:
        buckets = {}
        for r in records:
            buckets.setdefault(r[d], []).append(bool(r['hit']))
        tab = {b: {'n': len(v), 'recall': float(np.mean(v))}
               for b, v in buckets.items() if len(v) >= min_n}
        out[d] = tab
        if tab:
            b = min(tab, key=lambda k: tab[k]['recall'])
            worst[d] = (b, tab[b]['recall'])
    out['_worst'] = worst
    return out

rep = stratified_report(RECS, ['range_band', 'cls'])
assert rep['_worst']['cls'][0] == 'sign'
print('✅ 参考答案 4 通过')
print('   `min_n` 不是可选项：一个 3 样本的桶报出 0.000 的召回，')
print('   会让人去修一个不存在的问题（而这与 C10 的「样本量与置信区间」是同一条）。')

## 🧪 真实工程胶囊

```python
# ── 1) 先算天花板，再看排名（练习 3）──
#    做法：把预测直接设成真值，再给真值加上你估计的标注噪声，跑一遍你的评测代码。
gt_noisy = gt.copy()
gt_noisy[:, :3] += np.random.normal(0, SIGMA_ANN, (len(gt), 3))
ceiling = your_eval(predictions=gt, ground_truth=gt_noisy)   # ← 完美模型的上界
#    任何类别的 ceiling < 0.9，该类别的排名就不可信。

# ── 2) nuScenes 的口径：中心距离 + 误差分解 ──
from nuscenes.eval.detection.config import config_factory
cfg = config_factory('detection_cvpr_2019')
print(cfg.dist_ths)          # [0.5, 1.0, 2.0, 4.0] —— 中心距离，不是 IoU
#    上报 mAP 之外还有 ATE / ASE / AOE / AVE / AAE —— **不要只看 NDS 一个数**

# ── 3) 分割：绝不只报 mIoU（第 5 节）──
per_class_iou = confusion_iou(conf_mat)              # 逐类表
report = {
    'mIoU': per_class_iou.mean(),                    # 可比性
    'per_class': dict(zip(CLASS_NAMES, per_class_iou)),   # ← 唯一有信息量的
    'safety_critical_recall': {c: recall[c] for c in SAFETY_CLASSES},
}
#    频率加权 IoU 是明确错的口径（按频率而不是按后果加权）——不要报它

# ── 4) 分层维度按本课的六个来（第 7 节）──
STRATIFY_DIMS = ['range_band',        # 密度衰减 520 倍（模块 01）
                 'n_points_band',     # 临界点集只看 24 个点（模块 02）
                 'thickness_band',    # IoU 的三个乘子（模块 05）
                 'orientation_band',  # 锚框离散化（模块 04）
                 'class',             # 频率稀释（模块 05）
                 'has_vertical_overlap']  # BEV NMS（模块 04）
#    每一个维度都对应本课某一节量出的一个具体机制 —— 而不是随便分的
```

> **落地顺序建议**：先跑天花板审计（几行，可能直接告诉你「某个类别的指标没有意义」），
> 再把 mIoU 换成「mIoU + 逐类表 + 关键类召回」三件套，
> 最后按六个维度加分层。
> <em>而如果天花板审计发现某个类别不可信，先解决口径，再谈模型——
> 否则你在用一把分辨率不足的尺子做排名。</em>